In [1]:
import numpy as np
import bsf

ModuleNotFoundError: No module named 'bsf'

In [30]:
!wget https://ftp.ensembl.org/pub/release-116/fasta/homo_sapiens/pep/Homo_sapiens.GRCh38.pep.all.fa.gz
!gunzip Homo_sapiens.GRCh38.pep.all.fa.gz

--2026-07-10 16:13:07--  https://ftp.ensembl.org/pub/release-116/fasta/homo_sapiens/pep/Homo_sapiens.GRCh38.pep.all.fa.gz
Resolving ftp.ensembl.org (ftp.ensembl.org)... 193.62.193.169
Connecting to ftp.ensembl.org (ftp.ensembl.org)|193.62.193.169|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 23319936 (22M) [application/x-gzip]
Saving to: ‘Homo_sapiens.GRCh38.pep.all.fa.gz’

Homo_sapiens.GRCh38 100%[===================>]  22.24M  61.6MB/s    in 0.4s    

2026-07-10 16:13:08 (61.6 MB/s) - ‘Homo_sapiens.GRCh38.pep.all.fa.gz’ saved [23319936/23319936]



In [40]:
def load_fasta_as_dict(fasta_path):
    proteins = {}
    current_header = None
    current_sequence = []

    with open(fasta_path, "r") as fasta_file:
        for line in fasta_file:
            line = line.strip()

            if not line:
                continue

            if line.startswith(">"):
                # Save the previous sequence
                if current_header is not None:
                    proteins[current_header] = "".join(current_sequence)

                current_header = line[1:]  # remove the leading ">"
                current_sequence = []
            else:
                current_sequence.append(line)

        # Save the final sequence
        if current_header is not None:
            proteins[current_header] = "".join(current_sequence)

    return proteins

In [41]:
fasta_path = "Homo_sapiens.GRCh38.pep.all.fa"

all_proteins = load_fasta_as_dict(fasta_path)

print(len(all_proteins))

382428


In [43]:
list(all_proteins.values())[:10]

['MGSQVHLLSFLLLWISDTRAETTLTQSPAFMSATPGDKVNISCKASQDIDDDMNWYQQKPGEAAIFIIQEATTLVPGIPPRFSGSGYGTDFTLTINNIESEDAAYYFCLQHDNFP',
 'MDMRVPAQLLGLLLLWLPGAKCDIQMTQSPSTLSASVGDRVTITCRASQSISSWLAWYQQKPGKAPKLLIYKASSLESGVPSRFSGSGSGTEFTLTISSLQPDDFATYYCQQYNSYS',
 'MDMRVPAQLLGLLLLWLPGARCAIQMTQSPSSLSASVGDRVTITCRASQGIRNDLGWYQQKPGKAPKLLIYAASSLQSGVPSRFSGSGSGTDFTLTISSLQPEDFATYYCLQDYNYP',
 'MEAPAQLLFLLLLWLPDTTREIVMTQSPPTLSLSPGERVTLSCRASQSVSSSYLTWYQQKPGQAPRLLIYGASTRATSIPARFSGSGSGTDFTLTISSLQPEDFAVYYCQQDYNLP',
 'MRVPAQLLGLLLLWLPGARCAIRMTQSPSSFSASTGDRVTITCRASQGISSYLAWYQQKPGKAPKLLIYAASTLQSGVPSRFSGSGSGTDFTLTISCLQSEDFATYYCQQYYSYP',
 'MDMRVPAQLLGLLLLWLPGARCDIQLTQSPSFLSASVGDRVTITCRASQGISSYLAWYQQKPGKAPKLLIYAASTLQSGVPSRFSGSGSGTEFTLTISSLQPEDFATYYCQQLNSYP',
 'MEAPAQLLFLLLLWLPDTTGEIVLTQSPATLSLSPGERATLSCRASQSVSSYLAWYQQKPGQAPRLLIYDASNRATGIPARFSGSGSGTDFTLTISSLEPEDFAVYYCQQRSNWP',
 'MDMRVPAQLLGLLLLWFPGSRCDIQMTQSPSSVSASVGDRVTITCRASQGISSWLAWYQQKPGKAPKLLIYAASSLQSGVPSRFSGSGSGTDFTLTISSLQPEDFATYYCQQANSFP',
 'MEAPAQLLFLLLLWLPDTTGEIVMTQSPA

In [84]:
ten_proteins = dict(list(all_proteins.items())[:10000])

In [46]:
ten_proteins

{'ENSP00000487796.1 pep scaffold:GRCh38:HG2290_PATCH:6058:6610:1 gene:ENSG00000282172.1 transcript:ENST00000632585.1 gene_biotype:IG_V_gene transcript_biotype:IG_V_gene gene_symbol:IGKV5-2 description:immunoglobulin kappa variable 5-2 [Source:HGNC Symbol;Acc:HGNC:5835]': 'MGSQVHLLSFLLLWISDTRAETTLTQSPAFMSATPGDKVNISCKASQDIDDDMNWYQQKPGEAAIFIIQEATTLVPGIPPRFSGSGYGTDFTLTINNIESEDAAYYFCLQHDNFP',
 'ENSP00000488639.1 pep scaffold:GRCh38:HG2290_PATCH:56127:56783:-1 gene:ENSG00000282801.1 transcript:ENST00000632205.1 gene_biotype:IG_V_gene transcript_biotype:IG_V_gene gene_symbol:IGKV1-5 description:immunoglobulin kappa variable 1-5 [Source:HGNC Symbol;Acc:HGNC:5741]': 'MDMRVPAQLLGLLLLWLPGAKCDIQMTQSPSTLSASVGDRVTITCRASQSISSWLAWYQQKPGKAPKLLIYKASSLESGVPSRFSGSGSGTEFTLTISSLQPDDFATYYCQQYNSYS',
 'ENSP00000488804.1 pep scaffold:GRCh38:HG2290_PATCH:75088:75593:-1 gene:ENSG00000282163.1 transcript:ENST00000632891.1 gene_biotype:IG_V_gene transcript_biotype:IG_V_gene gene_symbol:IGKV1-6 description:immunoglo

In [89]:
standard_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")

ten_proteins = {
    protein_id: "".join(
        aa for aa in sequence.upper()
        if aa in standard_amino_acids
    )
    for protein_id, sequence in ten_proteins.items()
}

In [ ]:
import torch
import esm

# Load ESM-2 model
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval()  # disables dropout for deterministic results

# Prepare data (first 2 sequences from ESMStructuralSplitDataset superfamily / 4)
data = list(ten_proteins.items())

batch_labels, batch_strs, batch_tokens = batch_converter(data)
batch_lens = (batch_tokens != alphabet.padding_idx).sum(1)

# Extract per-residue representations (on CPU)
with torch.no_grad():
    results = model(batch_tokens, repr_layers=[33], return_contacts=False)
token_representations = results["representations"][33]

# Generate per-sequence representations via averaging
# NOTE: token 0 is always a beginning-of-sequence token, so the first residue is token 1.
sequence_representations = []
for i, tokens_len in enumerate(batch_lens):
    sequence_representations.append(token_representations[i, 1 : tokens_len - 1].mean(0))

In [57]:
sequence_representations = torch.stack(sequence_representations)
norm_sequence_representations = sequence_representations - sequence_representations.mean(dim=0)

tensor([[-0.0226, -0.0022, -0.0917,  ..., -0.0600, -0.0291,  0.0937],
        [-0.0571, -0.0457, -0.0726,  ..., -0.0937,  0.0383,  0.0916],
        [-0.0428, -0.0496, -0.0671,  ..., -0.0781,  0.0484,  0.1111],
        ...,
        [-0.0463, -0.0461, -0.0677,  ..., -0.0836,  0.0311,  0.1042],
        [-0.0521, -0.0352, -0.0601,  ..., -0.0904,  0.0251,  0.1052],
        [-0.0256, -0.0385, -0.0559,  ..., -0.0882,  0.0344,  0.1168]])

In [81]:
x = norm_sequence_representations
x = (x / np.sqrt((x**2).sum(1).mean()))*(np.sqrt(x.shape[1]))

In [83]:
model = bsf.GrassmannianBSF(d=x.shape[1], n_groups=256, group_size=3, l0=8)
bsf.train(model, x, epochs=300, lr=3e-3)

ZeroDivisionError: float division by zero